# SLURM scripts (Altamira)

In [5]:
from pathlib import Path
import pandas as pd

In [6]:
df_projects = pd.read_csv("./project_evaluation.csv")
df_projects.groupby("SubSet").size()

SubSet
J120    90
J30     90
J60     90
J90     90
dtype: int64

In [ ]:
# Create one SLURM array script per model for the final model evaluation.
# Each array task evaluates one project from project_evaluation.csv.

jobs_dir = Path("./jobs")
logs_dir = Path("./slurm_logs")
cluster_root = "/gpfs/users/res/uc27/uc27001/glassboxevm"
jobs_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)

selected_lhs = {
    "ebm": 24,
    "rf": 14,
    "xgb": 39,
    "sv": 0,
    "en": 36,
    "fastai": 41,  # re-run selection (2026-11, constant inputs dropped): best mean rank over the 50 LHS configurations
}

time_limits = {
    "rf": "00:30:00",
    "xgb": "00:30:00",
    "sv": "00:30:00",
    "en": "00:30:00",
    "ebm": "02:00:00",
    "fastai": "00:15:00",  # ~1 min per project on 1 CPU; short limit for fast scheduling
}

# Models whose script is (re)generated. Other scripts in jobs/ were edited by hand
# (e.g. EN: --mem=8G, %10) and must not be overwritten.
models_to_generate = ["fastai"]

# Extra SBATCH lines and environment setup per model
extra_sbatch = {
    "fastai": ["#SBATCH --mem=4G"],
}
extra_env = {
    # FastAI imports matplotlib (headless nodes) and writes caches/config files
    "fastai": [
        "export MPLBACKEND=Agg",
        "export CACHE_ROOT=/gpfs/users/res/uc27/uc27001/.cache",
        "export XDG_CACHE_HOME=${CACHE_ROOT}",
        "export FASTAI_HOME=${CACHE_ROOT}/fastai",
        "export TORCH_HOME=${CACHE_ROOT}/torch",
        "export MPLCONFIGDIR=${CACHE_ROOT}/matplotlib",
        "mkdir -p ${FASTAI_HOME} ${TORCH_HOME} ${MPLCONFIGDIR}",
    ],
}

array_last_index = len(df_projects) - 1
max_concurrent_tasks = 30

for model, lhs_id in selected_lhs.items():
    if model not in models_to_generate:
        continue
    script_path = jobs_dir / f"slurm_model_evaluation_{model}_lhs_{lhs_id}.sh"
    script_path.write_text(
        "#!/bin/bash\n"
        f"#SBATCH --job-name=meval_{model}_{lhs_id}\n"
        f"#SBATCH --output={cluster_root}/model_evaluation/slurm_logs/meval_{model}_{lhs_id}_%A_%a.out\n"
        f"#SBATCH --error={cluster_root}/model_evaluation/slurm_logs/meval_{model}_{lhs_id}_%A_%a.err\n"
        f"#SBATCH --time={time_limits[model]}\n"
        + "".join(f"{line}\n" for line in extra_sbatch.get(model, []))
        +
        f"#SBATCH --array=0-{array_last_index}%{max_concurrent_tasks}\n"
        "#SBATCH --nodes=1\n"
        "#SBATCH --ntasks=1\n"
        "#SBATCH --cpus-per-task=1\n"
        "#SBATCH --hint=nomultithread\n"
        f"#SBATCH --chdir={cluster_root}\n"
        "\n"
        + "".join(f"{line}\n" for line in extra_env.get(model, []))
        +
        "export OPENBLAS_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "export OMP_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "export MKL_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "export NUMEXPR_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "\n"
        "type module &>/dev/null && module purge\n"
        "source /gpfs/users/res/uc27/uc27001/mlenv/bin/activate\n"
        "\n"
        "PROJECT_NAME=$(sed -n \"$((SLURM_ARRAY_TASK_ID + 2))p\" "
        f"{cluster_root}/model_evaluation/project_evaluation.csv | cut -d, -f3 | sed 's/\\.sm$//')\n"
        "\n"
        "srun --ntasks=1 --cpus-per-task=${SLURM_CPUS_PER_TASK} --cpu-bind=cores "
        f"python3 ./model_evaluation/model_evaluation.py {model} {lhs_id} "
        "${PROJECT_NAME}\n",
        encoding="utf-8",
        newline="\n",
    )

sorted(jobs_dir.glob("slurm_model_evaluation_*.sh"))


**EBM job** changes

Slurm script:
```basch
#SBATCH --time=02:00:00
```

**EN job** changes

Slurm script:
```basch
#SBATCH --mem=8G
#SBATCH --array=0-359%10
```
model_evaluation.py:
```basch
max_iter=50000
```

**Run in Altamira**

Example: model RF for the hyperparametrization corresponding to LHS 14

```basch
cd /gpfs/users/res/uc27/uc27001/glassboxevm
sbatch model_evaluation/jobs/slurm_model_evaluation_rf_lhs_14.sh
```